# 004 — PSHA results: hazard curves

Post-processes the six WP1 PSHA calculations set up and run by `003-psha_setup_and_analyses.ipynb`
(**AvgSA 0–3** and **AvgSA 0–6**, each at GMM truncation level **3, 4 and 5 sigma**) into
hazard curves for the 60 selected sites.

1. Resolves the OpenQuake `calc_id` of each analysis from `wp1/psha_manifest.json` — no
   hardcoded integers.
2. Extracts the hazard curves from each datastore into one flat dictionary per analysis,
   in **mean annual frequency of exceedance (MAFE)**.
3. Saves the six dictionaries as pickles suffixed `_3sig`, `_4sig`, `_5sig`.
4. Plots the hazard curves by region, contrasting **high** and **low/moderate** seismicity sites.
5. Plots the effect of the **GMM truncation level** on the individual site curves.

## Prerequisites

`003-psha_setup_and_analyses.ipynb` must have been run with `DRY_RUN = False`, so that:

- all six entries exist in `hazard_models/eshm20/wp1/psha_manifest.json`, and
- the corresponding OpenQuake datastores (`calc_<id>.hdf5`) are present in the local
  `oqdata` directory.

The datastores are **machine-local and not tracked** (they are large and rebuildable). The
manifest is the git-tracked pointer that makes them reproducible — if the datastores are
missing, re-run `003` rather than editing calc ids here.

## Dependencies

**Upstream:** `001-site_selection.ipynb` → `results/01_site_selection/sites.csv` (the site
register, and the source of the `seismicity` / `region` grouping used in the plots);
`003-psha_setup_and_analyses.ipynb` → `wp1/psha_manifest.json` and `wp1/site_model_all_sites.csv`.

**Downstream:** `017-disagg_imls_for_msa_stripes.ipynb` consumes the `_3sig` / `_4sig` /
`_5sig` pickles written here to pick the MSA stripe IMLs.

## Outputs

Six pickles in `data_processed/03_site_hazard` (**DVC-tracked**, like the rest of that folder):

```
AvgSA_03_hazard_curves_60sites_{3,4,5}sig.pickle
AvgSA_06_hazard_curves_60sites_{3,4,5}sig.pickle
```

All six share **one** structure — the raw `get_hcurves_from_dstore` shape:

```
hcs[site_id][imt][stat] -> np.ndarray, shape (n_iml, 2)
                           column 0 = IML [g]
                           column 1 = MAFE [1/yr]
```

`site_id` is the positional row index of `sites.csv` (0–59). There is no
`(seismicity, region)` grouping level and no `"mean_site"` key — grouping is applied at
plot time from `sites.csv`, so the artifacts stay flat and uniform.

In [ ]:
%load_ext autoreload
%autoreload 2

## 0. Setup & parameters

In [ ]:
import string
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.lines as mlines

from openquake.commonlib.datastore import read

from phd_project.config import config
from phd_project.plotting.plotting import custom_log_formatter
from phd_project.scripts.oqhelpers import get_hcurves_from_dstore
from phd_project.scripts.WP1_ground_motion_set import oq_runner
from phd_project.scripts.WP1_ground_motion_set.hazard import get_mask

cfg = config.load_config()

In [ ]:
# -----------------------------------------------------------------------------
# PARAMETERS
# -----------------------------------------------------------------------------
# IM definition -> axis label. The key is the prefix of both the manifest entry
# name and the output pickle name.
IMS = {"AvgSA_03": "AvgSA[0,3]", "AvgSA_06": "AvgSA[0,6]"}

# GMM truncation level: manifest suffix -> output-filename suffix.
TRUNCS = {"eps3": "3sig", "eps4": "4sig", "eps5": "5sig"}

IMT = "AvgSA"   # the single IMT present in these calculations
STAT = "mean"   # the OpenQuake statistic plotted (the pickles keep all of them)

N_SITES = 60
REGIONS = list(range(6))
SEISMICITIES = ("high", "lowmod")

HAZ_DIR = cfg["proc_data"]["site_hazard"]
MANIFEST_FP = cfg["hazard_models"]["eshm20_wp1_psha_manifest"]
SITES_FP = cfg["results"]["selected_sites_csv"]
SITE_MODEL_FP = cfg["hazard_models"]["eshm20_wp1_site_model"]

# SAVE: write the pickles. False = extract and plot only, touching nothing on disk.
SAVE = True
# -----------------------------------------------------------------------------

print(f"hazard dir: {HAZ_DIR}")
print(f"manifest:   {MANIFEST_FP}")
print(f"analyses:   {len(IMS) * len(TRUNCS)} "
      f"({sorted(IMS)} x {sorted(TRUNCS)})")
print(f"SAVE={SAVE}")

## 1. Sites & calculation ids

`sites.csv` and `wp1/site_model_all_sites.csv` are row-aligned by construction (`003`
writes the latter from the former), so they can be joined positionally. The resulting
index **is** the site id used everywhere in this project.

The two grouping keys come from `sites.csv`: `seismicity` (`"high"` / `"lowmod"`) and
`region` (`0`–`5`) — 5 sites in each of the 12 combinations.

In [ ]:
sel_sites = pd.read_csv(SITES_FP)
site_model = pd.read_csv(SITE_MODEL_FP)
site_metadata = pd.concat([sel_sites, site_model], axis=1).T.drop_duplicates().T

assert len(site_metadata) == N_SITES, (
    f"expected {N_SITES} sites, got {len(site_metadata)} — "
    f"check {SITES_FP} and {SITE_MODEL_FP} are the same 60 rows in the same order")

print(site_metadata.groupby(["seismicity", "region"]).size().to_string())

In [ ]:
calc_ids = oq_runner.load_calc_ids(MANIFEST_FP)

expected = [f"{im}_psha_{eps}" for im in IMS for eps in TRUNCS]
missing = [name for name in expected if name not in calc_ids]
assert not missing, (
    f"missing from {MANIFEST_FP}: {missing} — re-run 003 with DRY_RUN = False")

for name in expected:
    print(f"{name:24s} calc_id = {calc_ids[name]}")

## 2. Extract & save hazard curves

One pass over the six datastores. `get_hcurves_from_dstore` converts the OpenQuake
probability of exceedance to MAFE (`-ln(1 - poe) / investigation_time`), so the curves are
in MAFE as stored.

In [ ]:
hcs = {}   # hcs[(im, sig)] -> hazard curves for one analysis

for im in IMS:
    for eps, sig in TRUNCS.items():
        dstore = read(calc_ids[f"{im}_psha_{eps}"])
        curves = get_hcurves_from_dstore(dstore, mafe=True)

        assert len(curves) == N_SITES, (
            f"{im}/{eps}: datastore has {len(curves)} sites, expected {N_SITES}")
        hcs[(im, sig)] = curves

        if SAVE:
            fp = HAZ_DIR / f"{im}_hazard_curves_60sites_{sig}.pickle"
            with open(fp, "wb") as f:
                pickle.dump(curves, f)
            print(f"wrote {fp.name}")

print(f"\n{len(hcs)} analyses loaded from openquake and processed.")

In [ ]:
# Structure check: every analysis must have the same flat, MAFE shape.
for (im, sig), curves in hcs.items():
    assert sorted(curves) == list(range(N_SITES)), f"{im}/{sig}: site keys are not 0..{N_SITES-1}"
    assert IMT in curves[0], f"{im}/{sig}: IMT {IMT!r} not in {list(curves[0])}"
    assert STAT in curves[0][IMT], f"{im}/{sig}: stat {STAT!r} not in {list(curves[0][IMT])}"

    hc = curves[0][IMT][STAT]
    assert hc.ndim == 2 and hc.shape[1] == 2, f"{im}/{sig}: curve shape {hc.shape}, expected (n, 2)"
    assert (np.diff(hc[:, 1]) <= 0).all(), f"{im}/{sig}: MAFE column is not decreasing"

print(f"OK — {len(hcs)} x hcs[site_id][{IMT!r}][stat] -> (n_iml, 2), "
      f"col 1 = MAFE, stats = {list(hcs[('AvgSA_03', '3sig')][0][IMT])}")

## 3. Plotting constants & helpers

The flat structure carries no group means, so the group mean is computed at plot time.
It is the same quantity the old grouped pickles stored: the mean over the **MAFE column**
of the sites in a group, against the shared IML column.

In [ ]:
legend_font_params = {'size': 9}
legend_title_font_params = {'weight': 'bold', 'size': 9}

# hazard curve plots (high vs low/mod)
hc_labels = ["High site", "Low/mod site", "Mean - high sites", "Mean - low/mod sites"]
hc_colours = ["r", "g", "r", "g"]
hc_line_styles = ["-", "-", "--", "--"]
hc_alphas = [0.3, 0.3, 1.0, 1.0]

# truncation level comparison: colour = site slot within a region, style = truncation
site_colours = ["C0", "C1", "C2", "C3", "C4"]
trunc_line_styles = {"3sig": "-", "4sig": "--", "5sig": ":"}

seismicity_labels = {"high": "High seismicity", "lowmod": "Low/moderate seismicity"}

In [ ]:
def group_site_ids(metadata, seismicity, region):
    """Site ids belonging to one (seismicity, region) group, in site-id order."""
    mask = get_mask(["seismicity", "region"], [seismicity, region], metadata)
    return sorted(metadata[mask].index.tolist())


def mean_curve(curves, site_ids, imt=IMT, stat=STAT):
    """Mean hazard curve over `site_ids` -> ndarray (n_iml, 2).

    The IMLs are identical across sites within a calculation, so only the MAFE
    column is averaged.
    """
    imls = curves[site_ids[0]][imt][stat][:, 0]
    mafes = np.vstack([curves[i][imt][stat][:, 1] for i in site_ids])
    return np.vstack([imls, mafes.mean(axis=0)]).T


def style_hc_axis(ax, panel_idx, region):
    """The shared hazard-curve axis furniture."""
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(custom_log_formatter))
    ax.set_xlim(0.001, 5)
    ax.set_ylim(1e-6, 1)

    ax.grid(True, which="both", ls="-.", color="0.8")
    ax.minorticks_on()
    ax.tick_params(axis='y', which='minor', left=False)

    ax.text(0.0011, 1.5e-6, f"({string.ascii_uppercase[panel_idx]})")
    ax.text(0.5, 2e-1, f"Region {region}")

## 4. Hazard curves by region — high vs low/moderate seismicity

One figure per (IM definition × truncation level). Faint solid lines are individual sites,
dashed lines the group mean; red is high seismicity, green low/moderate.

In [ ]:
for im, im_label in IMS.items():
    for sig in TRUNCS.values():
        curves = hcs[(im, sig)]

        fig, axs = plt.subplots(2, 3, sharex=True, sharey=True, figsize=(9, 6))

        for ii, (r, ax) in enumerate(zip(REGIONS, axs.flatten())):
            for s in SEISMICITIES:
                color = "r" if s == "high" else "g"
                site_ids = group_site_ids(site_metadata, s, r)

                for site_id in site_ids:
                    hc = curves[site_id][IMT][STAT]
                    ax.loglog(hc[:, 0], hc[:, 1], ls="-", color=color, alpha=0.3)

                hc = mean_curve(curves, site_ids)
                ax.loglog(hc[:, 0], hc[:, 1], ls="--", color=color)

            style_hc_axis(ax, ii, r)

        axs[0, 0].set_ylabel("MAFE [1/yr]")
        axs[1, 0].set_ylabel("MAFE [1/yr]")
        for ax in axs[1, :]:
            ax.set_xlabel(f"{im_label} [g]")

        handles = [
            mlines.Line2D([], [], color=c, label=l, linestyle=ls, alpha=a)
            for c, l, ls, a in zip(hc_colours, hc_labels, hc_line_styles, hc_alphas)
        ]

        leg = fig.legend(handles=handles,
                         ncols=4,
                         loc="upper left",
                         bbox_to_anchor=(0.075, 0.955),
                         prop=legend_font_params,
                         frameon=False)
        leg._legend_box.align = "left"

        fig.suptitle(f"{im_label} - Hazard Curves for Selected Sites "
                     f"({sig[0]}$\\sigma$ truncation)")
        fig.tight_layout()

## 5. Effect of the GMM truncation level

One figure per (IM definition × seismicity class). Within each region panel the same five
sites are drawn at all three truncation levels: **colour** identifies the site's slot
within its region, **line style** the truncation level.

The truncation level sets a ceiling on the ground motion the GMM will produce, which is
what makes each curve fall away at its right-hand end. Raising it extends the curve into
rarer, larger ground motions.

In [ ]:
for im, im_label in IMS.items():
    for s in SEISMICITIES:

        fig, axs = plt.subplots(2, 3, sharex=True, sharey=True, figsize=(9, 6))

        for ii, (r, ax) in enumerate(zip(REGIONS, axs.flatten())):
            site_ids = group_site_ids(site_metadata, s, r)

            for j, site_id in enumerate(site_ids):
                for sig in TRUNCS.values():
                    hc = hcs[(im, sig)][site_id][IMT][STAT]
                    ax.loglog(hc[:, 0], hc[:, 1],
                              color=site_colours[j],
                              ls=trunc_line_styles[sig],
                              lw=1.2)

            style_hc_axis(ax, ii, r)

        axs[0, 0].set_ylabel("MAFE [1/yr]")
        axs[1, 0].set_ylabel("MAFE [1/yr]")
        for ax in axs[1, :]:
            ax.set_xlabel(f"{im_label} [g]")

        site_handles = [
            mlines.Line2D([], [], color=c, label=f"Site {j + 1}")
            for j, c in enumerate(site_colours)
        ]
        trunc_handles = [
            mlines.Line2D([], [], color="0.3", ls=ls, label=f"{sig[0]}$\\sigma$")
            for sig, ls in trunc_line_styles.items()
        ]

        leg1 = fig.legend(handles=site_handles, title="Site in region",
                          ncols=5, loc="upper left", bbox_to_anchor=(0.075, 0.945),
                          prop=legend_font_params,
                          title_fontproperties=legend_title_font_params,
                          frameon=False)
        leg1._legend_box.align = "left"

        leg2 = fig.legend(handles=trunc_handles, title="Truncation",
                          ncols=3, loc="upper left", bbox_to_anchor=(0.60, 0.945),
                          prop=legend_font_params,
                          title_fontproperties=legend_title_font_params,
                          frameon=False)
        leg2._legend_box.align = "left"

        fig.suptitle(f"{im_label} - {seismicity_labels[s]} sites: "
                     f"effect of GMM truncation")
        fig.tight_layout(rect=(0, 0, 1, 0.92))